# State-Conditioned Entropy Pooling

A clean, offline reference workflow for the **predictive conditioning layer** extracted from the v20–v26 research sequence.

This notebook deliberately does **not** reproduce the physical cross-asset scenario engine. The intended workflow is:

```text
Multi-Asset Scenario Engine
        ↓ physical scenario support + repriced instrument returns
State-Conditioned Entropy Pooling
        ↓ state-conditioned prior + probabilistic views
posterior VaR / ES / tail attribution / min-CVaR
```

The example uses a synthetic scenario support so it runs without market-data downloads. Replace that support with output from the separate `multi-asset-scenario-engine` repository for the live workflow.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from state_conditioned_entropy_pooling import (
    EventView,
    ScenarioSupport,
    event_uplift,
    uplift_target,
    posterior_summary,
    solve_event_views,
    state_conditioned_prior,
    view_set_diagnostics,
    weighted_min_cvar,
    weighted_var_es,
)

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
RNG = np.random.default_rng(42)


## 1. Build a physical scenario support

In the live project these rows come from the cross-asset scenario engine **after horizon repricing**. Here we generate an offline four-sleeve support with a shared systematic shock and idiosyncratic noise.


In [2]:
n_scen = 8_000
common = RNG.standard_t(df=6, size=n_scen)

instrument_returns = pd.DataFrame({
    "equity":  0.002 + 0.015 * common + 0.012 * RNG.normal(size=n_scen),
    "credit":  0.001 + 0.006 * common + 0.006 * RNG.normal(size=n_scen),
    "rates":   0.000 - 0.003 * common + 0.004 * RNG.normal(size=n_scen),
    "fx":      0.000 + 0.004 * common + 0.008 * RNG.normal(size=n_scen),
})

scenario_features = pd.DataFrame({
    "equity_loss_event": (instrument_returns["equity"] <= -0.030).astype(float),
    "credit_widen_event": (instrument_returns["credit"] <= -0.014).astype(float),
})

uniform_prior = np.full(n_scen, 1.0 / n_scen)
support = ScenarioSupport(instrument_returns, scenario_features, prior=uniform_prior)
support.instrument_returns.describe().T


,count,mean,std,min,25%,50%,75%,max
equity,"8,000.000000",0.001938,0.021646,-0.110314,-0.011811,0.001850,0.015659,0.136756
credit,"8,000.000000",0.000843,0.009313,-0.052742,-0.005241,0.000742,0.006867,0.048286
rates,"8,000.000000",-0.000015,0.005353,-0.023557,-0.003473,-0.000032,0.003504,0.023996
fx,"8,000.000000",-0.000177,0.009307,-0.037488,-0.006430,-0.000121,0.006020,0.037256


## 2. State-conditioned Flexible-Probabilities prior

Historical horizon windows are weighted by the distance between the **state known immediately before each historical window** and the current state. The example uses three standardized coordinates: market turbulence, growth / credit-cycle stress and inflation / term-premium pressure.


In [3]:
n_hist = n_scen
historical_states = RNG.normal(size=(n_hist, 3))
current_state = np.array([1.40, 1.10, 0.45])

ffp_prior = state_conditioned_prior(
    historical_states,
    current_state,
    fraction=0.10,
    min_count=100,
    bandwidth_floor=0.35,
)

pd.Series({
    "uniform_effective_n": 1 / np.sum(uniform_prior**2),
    "ffp_effective_n": 1 / np.sum(ffp_prior**2),
    "ffp_max_weight": ffp_prior.max(),
})


uniform_effective_n   8,000.000000
ffp_effective_n       4,386.684007
ffp_max_weight            0.000493
dtype: float64

## 3. Channel-specific event views

The v23 redesign separated information channels instead of sending many signals through one generic view funnel. A deadband prevents small analogue-estimation noise from activating a costly posterior view.


In [4]:
credit_signal_hist = historical_states[:, 1] + 0.25 * RNG.normal(size=n_hist)
equity_signal_hist = historical_states[:, 0] + 0.25 * RNG.normal(size=n_hist)

credit_event_hist = (
    0.55 * credit_signal_hist + 0.30 * historical_states[:, 0] + RNG.normal(size=n_hist) > 1.25
).astype(float)
equity_event_hist = (
    0.60 * equity_signal_hist + 0.20 * historical_states[:, 1] + RNG.normal(size=n_hist) > 1.35
).astype(float)

credit_uplift = event_uplift(credit_signal_hist, 1.30, credit_event_hist, min_count=100)
equity_uplift = event_uplift(equity_signal_hist, 1.45, equity_event_hist, min_count=100)

prior_credit = float(ffp_prior @ scenario_features["credit_widen_event"].to_numpy())
prior_equity = float(ffp_prior @ scenario_features["equity_loss_event"].to_numpy())

targets = {
    "credit_widen_event": uplift_target(prior_credit, credit_uplift, deadband=0.05),
    "equity_loss_event": uplift_target(prior_equity, equity_uplift, deadband=0.05),
}

pd.DataFrame({
    "prior_probability": [prior_credit, prior_equity],
    "uplift": [credit_uplift, equity_uplift],
    "target_probability": [targets["credit_widen_event"], targets["equity_loss_event"]],
}, index=["credit_widen_event", "equity_loss_event"])


,prior_probability,uplift,target_probability
credit_widen_event,0.053070,1.951043,0.103543
equity_loss_event,0.067204,2.175718,0.146217


## 4. Entropy-pooling posterior

Entropy Pooling keeps the scenario support fixed and finds the **minimum-KL posterior** satisfying the active views.


In [5]:
views = [
    EventView("credit_widen_event", scenario_features["credit_widen_event"].to_numpy(), targets["credit_widen_event"]),
    EventView("equity_loss_event", scenario_features["equity_loss_event"].to_numpy(), targets["equity_loss_event"]),
]

ep_result = solve_event_views(ffp_prior, views)
posterior = ep_result.probabilities

pd.Series({
    "solver_success": ep_result.success,
    "eq_max_abs_residual": ep_result.eq_max_abs_residual,
    "ineq_max_violation": ep_result.ineq_max_violation,
    **posterior_summary(ffp_prior, posterior),
})


solver_success                      True
eq_max_abs_residual             0.000000
ineq_max_violation              0.000000
relative_entropy                0.043617
prior_effective_n           5,359.118314
posterior_effective_n       5,117.736289
posterior_effective_ratio       0.639717
max_probability                 0.001269
max_prior_ratio                 2.785536
dtype: object

## 5. View-set non-redundancy

The v26 research branch added an ARPM-style sanity check on the prior-measure covariance of the view variables. This is distinct from posterior-tilt rank: a view set may be non-redundant in principle while actual signals still map into similar posterior tilts.


In [6]:
view_rows = np.vstack([
    scenario_features["credit_widen_event"].to_numpy(),
    scenario_features["equity_loss_event"].to_numpy(),
])

diag = view_set_diagnostics(view_rows, ffp_prior, threshold_fraction=0.5)
pd.Series({k: v for k, v in diag.items() if k != "covariance"})


n_views                      2
algebraic_rank               2
effective_rank        1.877809
participation_ratio   1.780293
threshold             1.000000
passes                    True
dtype: object

## 6. Posterior portfolio tail risk

Posterior probabilities feed directly into VaR/ES and weighted min-CVaR without re-running the repricer.


In [7]:
portfolio_weights = pd.Series({"equity": 0.45, "credit": 0.30, "rates": 0.15, "fx": 0.10})
portfolio_returns = instrument_returns @ portfolio_weights

pd.DataFrame({
    "uniform_prior": weighted_var_es(portfolio_returns, uniform_prior, alpha=0.05),
    "state_FFP_prior": weighted_var_es(portfolio_returns, ffp_prior, alpha=0.05),
    "FFP_plus_EP_views": weighted_var_es(portfolio_returns, posterior, alpha=0.05),
}).T


,mean_return,volatility,VaR_95_loss,ES_95_loss,tail_mass
uniform_prior,0.001105,0.011695,0.017958,0.024161,0.050000
state_FFP_prior,0.001008,0.011643,0.018090,0.024672,0.050000
FFP_plus_EP_views,-0.001262,0.013248,0.024284,0.030803,0.050000


In [8]:
min_cvar_weights, _, min_cvar_info = weighted_min_cvar(
    instrument_returns, posterior, alpha=0.05, max_weight=0.60
)
display(min_cvar_weights.to_frame("posterior_min_CVaR_weight"))
display(min_cvar_info.to_frame("value"))


,posterior_min_CVaR_weight
equity,0.000000
credit,0.240180
rates,0.600000
fx,0.159820


,value
LP objective,0.006467
zeta,0.005174
max weight,0.600000
sum weights,1.000000
mean_return,-0.000012
volatility,0.003109
VaR_95_loss,0.005174
ES_95_loss,0.006467
tail_mass,0.050000


## 7. 2×2 conditioning design

The research question is explicitly factorial:

```text
                     views OFF          views ON
uniform prior        baseline           view-only
state FFP prior      prior-only         prior + views
```

The live walk-forward research scores these arms using VaR coverage, ES residual tests, FZ0 loss and Diebold-Mariano inference. The static table below shows the four current-date distributions.


In [9]:
factorial = {}
for prior_name, q in {"uniform": uniform_prior, "state_FFP": ffp_prior}.items():
    factorial[(prior_name, "views_off")] = q
    factorial[(prior_name, "views_on")] = solve_event_views(q, views).probabilities

rows = []
for (prior_name, view_name), p in factorial.items():
    stats = weighted_var_es(portfolio_returns, p, alpha=0.05)
    rows.append({
        "prior": prior_name,
        "views": view_name,
        "ES95_loss": stats["ES_95_loss"],
        "VaR95_loss": stats["VaR_95_loss"],
        **posterior_summary(uniform_prior, p),
    })

pd.DataFrame(rows).set_index(["prior", "views"])


ES95_loss  VaR95_loss  ...  max_probability  max_prior_ratio
prior     views                             ...                                  
uniform   views_off   0.024161    0.017958  ...         0.000125         1.000000
          views_on    0.030218    0.023835  ...         0.000351         2.809236
state_FFP views_off   0.024672    0.018090  ...         0.000493         3.943892
          views_on    0.030803    0.024284  ...         0.001269        10.152722

[4 rows x 8 columns]

## 8. Integration boundary

Replace the synthetic `instrument_returns` and `scenario_features` with row-aligned outputs from the separate `multi-asset-scenario-engine` repository.

```text
scenario engine:
    generate states → reprice instruments → scenario P&Ls

this repository:
    construct prior → map signals into views → entropy pool
    → posterior risk → OOS factorial scoring
```

The older `entropy-pooling-stress-testing` repository remains separate as an explicit **stress-view engine**. This repository is a **predictive state-conditioning research layer**.


## 9. Research-status note

The v20–v26 sequence deliberately retained negative evidence. A preregistered cross-portfolio ranking prediction in the production factorial did **not** hold in the observed sample.

The correct claim for this repository is methodological:

> it builds and tests a disciplined state-conditioned probability layer over physical scenarios.

It does **not** claim universal predictive superiority of Entropy Pooling or Flexible Probabilities.
